In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Use CUDA if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
from pathlib import Path

text = Path("../../../data/tiny-shakespeare.txt").read_text()

In [3]:
print(text[0:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [4]:
class CharTokenizer:
    def __init__(self, vocabulary):
        self.token_id_for_char = {
            char: token_id for token_id, char in enumerate(vocabulary)
        }
        self.char_for_token_id = {
            token_id: char for token_id, char in enumerate(vocabulary)
        }

    @staticmethod
    def train_from_text(text):
        vocabulary = set(text)
        return CharTokenizer(sorted(list(vocabulary)))

    def encode(self, text):
        token_ids = []
        for char in text:
            token_ids.append(self.token_id_for_char[char])
        return torch.tensor(token_ids, dtype=torch.long)

    def decode(self, token_ids):
        chars = []
        for token_id in token_ids.tolist():
            chars.append(self.char_for_token_id[token_id])
        return "".join(chars)

    def vocabulary_size(self):
        return len(self.token_id_for_char)

In [5]:
tokenizer = CharTokenizer.train_from_text(text)

In [6]:
print(tokenizer.encode("Hello world"))
print(tokenizer.decode(tokenizer.encode("Hello world")))

tensor([20, 43, 50, 50, 53,  1, 61, 53, 56, 50, 42])
Hello world


In [7]:
print(f"Vocabulary size: {tokenizer.vocabulary_size()}")

Vocabulary size: 65


In [8]:
from torch.utils.data import Dataset


class TokenIdsDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, pos):
        assert pos < len(self.data) - self.block_size

        x = self.data[pos : pos + self.block_size]
        y = self.data[pos + 1 : pos + 1 + self.block_size]
        return x, y

In [9]:
config = {
    "vocabulary_size": tokenizer.vocabulary_size(),
    "context_size": 256,
    "d_embed": 768,
    "heads_num": 12,
    "layers_num": 10,
    "dropout_rate": 0.1,
    "use_bias": False,
}

config["head_size"] = config["d_embed"] // config["heads_num"]

In [10]:
class AttentionHead(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.Q_weights = nn.Linear(
            config["d_embed"], config["head_size"], config["use_bias"]
        )
        self.K_weights = nn.Linear(
            config["d_embed"], config["head_size"], config["use_bias"]
        )
        self.V_weights = nn.Linear(
            config["d_embed"], config["head_size"], config["use_bias"]
        )

        self.dropout = nn.Dropout(config["dropout_rate"])

        casual_attention_mask = torch.tril(
            torch.ones(config["context_size"], config["context_size"])
        )
        self.register_buffer("casual_attention_mask", casual_attention_mask)

    def forward(self, input):
        batch_size, tokens_num, d_embed = input.shape
        Q = self.Q_weights(input)
        K = self.K_weights(input)
        V = self.V_weights(input)

        attention_scores = Q @ K.transpose(1, 2)
        attention_scores = attention_scores.masked_fill(
            self.casual_attention_mask[:tokens_num, :tokens_num] == 0, -torch.inf
        )
        attention_scores = attention_scores / (K.shape[-1] ** 0.5)
        attention_scores = torch.softmax(attention_scores, dim=-1)
        attention_scores = self.dropout(attention_scores)

        return attention_scores @ V

In [11]:
input = torch.rand(8, config["context_size"], config["d_embed"])

In [12]:
ah = AttentionHead(config)

In [13]:
output = ah(input)

In [14]:
output.shape

torch.Size([8, 256, 64])

In [15]:
class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()

        heads_list = [AttentionHead(config) for _ in range(config["heads_num"])]
        self.heads = nn.ModuleList(heads_list)

        self.linear = nn.Linear(config["d_embed"], config["d_embed"])
        self.dropout = nn.Dropout(config["dropout_rate"])

    def forward(self, input):
        heads_outputs = [head(input) for head in self.heads]

        scores_change = torch.cat(heads_outputs, dim=-1)

        scores_change = self.linear(scores_change)
        return self.dropout(scores_change)

In [16]:
mha = MultiHeadAttention(config)

In [17]:
input = torch.rand(8, config["context_size"], config["d_embed"])

In [18]:
output = mha(input)

In [19]:
output.shape

torch.Size([8, 256, 768])

In [20]:
class FeedForward(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.linear_layers = nn.Sequential(
            nn.Linear(config["d_embed"], config["d_embed"] * 4),
            nn.GELU(),
            nn.Linear(config["d_embed"] * 4, config["d_embed"]),
            nn.Dropout(config["dropout_rate"]),
        )

    def forward(self, input):
        return self.linear_layers(input)

In [21]:
ff = FeedForward(config)

In [22]:
input = torch.rand(8, config["context_size"], config["d_embed"])

In [23]:
ouptut = ff(input)

In [24]:
output.shape

torch.Size([8, 256, 768])

In [25]:
class Block(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.multi_head = MultiHeadAttention(config)
        self.layer_norm_1 = nn.LayerNorm(config["d_embed"])

        self.feed_forward = FeedForward(config)
        self.layer_norm_2 = nn.LayerNorm(config["d_embed"])

    def forward(self, input):
        residual = input
        x = self.multi_head(self.layer_norm_1(input))
        x = x + residual

        residual = x
        x = self.feed_forward(self.layer_norm_2(x))
        return x + residual

In [26]:
b = Block(config)

In [27]:
ouptut = b(input)

In [28]:
output.shape

torch.Size([8, 256, 768])

In [29]:
class DemoGPT(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.token_embedding_layer = nn.Embedding(
            config["vocabulary_size"], config["d_embed"]
        )
        self.positional_embedding_layer = nn.Embedding(
            config["context_size"], config["d_embed"]
        )

        blocks = [Block(config) for _ in range(config["layers_num"])]
        self.layers = nn.Sequential(*blocks)

        self.layer_norm = nn.LayerNorm(config["d_embed"])
        self.unembedding = nn.Linear(
            config["d_embed"], config["vocabulary_size"], bias=False
        )

    def forward(self, token_ids):
        batch_size, tokens_num = token_ids.shape

        x = self.token_embedding_layer(token_ids)
        sequence = torch.arange(tokens_num, device=device)
        x = x + self.positional_embedding_layer(sequence)

        x = self.layers(x)
        x = self.layer_norm(x)
        x = self.unembedding(x)

        return x

In [30]:
model = DemoGPT(config).to(device)

In [31]:
output = model(tokenizer.encode("Hi").unsqueeze(dim=0).to(device))

In [32]:
output.shape

torch.Size([1, 2, 65])

In [33]:
def generate(model, prompt_ids, max_tokens):
    output_ids = prompt_ids
    for _ in range(max_tokens):
        if output_ids.shape[1] >= config["context_size"]:
            break
        with torch.no_grad():
            logits = model(output_ids)

        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)
        # Sample a random token given the softmax distribution
        next_token_id = torch.multinomial(probs, num_samples=1)
        # Add new token to the output, and repeat the process
        output_ids = torch.cat([output_ids, next_token_id], dim=-1)
    return output_ids

In [34]:
def generate_with_prompt(model, tokenizer, prompt, max_tokens=100):
    model.eval()

    prompt = tokenizer.encode(prompt).unsqueeze(dim=0).to(device)

    return tokenizer.decode(generate(model, prompt, max_tokens=max_tokens)[0])

In [35]:
generate_with_prompt(model, tokenizer, "First Citizen:\n")

"First Citizen:\nLzIdbZ$s w:tDqO.-:mLKlIfB-q.JcDzDXgOi-Bmu;BTUycurwE-H X;puWHiML!s$X,o3.lMfIjXE$jd&zxWQCMo'f-EDL:FWed"

In [36]:
batch_size = 64

train_iterations = 500
evaluation_interval = 10
learning_rate = 4e-4
train_split = 0.9

In [37]:
# Step 1 - Split Data into Training and Validation Dataset

tokenized_text = tokenizer.encode(text).to(device)
# Get number of tokens in the training dataset. Should be train_split * number_of_tokens
train_count = int(train_split * len(tokenized_text))
# Split data into training and validation datasets
train_data, validation_data = tokenized_text[:train_count], tokenized_text[train_count:]

In [38]:
# Step 2 - Create Validation Dataset

train_dataset = TokenIdsDataset(train_data, config["context_size"])
# Create a validation dataset from the validation data
validation_dataset = TokenIdsDataset(validation_data, config["context_size"])

In [39]:
# Step 3 - Create Validation DataLoader

from torch.utils.data import Dataset, DataLoader, RandomSampler

train_sampler = RandomSampler(
                train_dataset, num_samples=batch_size * train_iterations, replacement=True
                )
train_dataloader = DataLoader(
                train_dataset, batch_size=batch_size, sampler=train_sampler
                )

validation_sampler = RandomSampler(validation_dataset, replacement=True)
# Create validation data loader
validation_dataloader = DataLoader(validation_dataset, batch_size=batch_size, sampler=validation_sampler)

In [40]:
# Step 4 - Calculate Validation Loss


# Compute validation loss for the model using "batches_num" batches
# from the validation data loader
@torch.no_grad()
def calculate_validation_loss(model, batches_num):
    model.eval()
    total_loss = 0

    # An iterator for the validation data loader
    validation_iter = iter(validation_dataloader)

    for _ in range(batches_num):
        idx, targets = next(validation_iter)
        logits = model(idx)

        # Call "next" function to get input and targets from the iterator
        input, targets = next(validation_iter)
        # Using the model compute logits given the input
        logits = model(input)

        # Use the "view" method to convert logits and targets so we could use the "cross_entropy" function
        logits_view = logits.view(batch_size * config["context_size"], config["vocabulary_size"])
        targets_view =targets.view(batch_size * config["context_size"])

        # Calculate cross entropy using logits and target data
        loss = F.cross_entropy(logits_view, targets_view)

        # Add loss to the "total_loss" variable
        # Note: we need to use the "item()" method to convert a tensor to a number
        total_loss += loss.item()
        

    average_loss = total_loss / batches_num

    return average_loss

In [41]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

In [42]:
# Step 5 - Update the Training Loop

import os
from IPython.display import display, clear_output
from matplotlib import pyplot as plt
from IPython.display import display
import ipywidgets as widgets
%matplotlib inline

plot_output = widgets.Output()

display(plot_output)

def update_plot(train_losses, train_steps, validation_losses, validation_steps):

  with plot_output:
    clear_output(wait=True)  # Clear only the plot output, not the text
    plt.figure(figsize=(7, 5))
    plt.plot(train_steps, train_losses, label='Training Loss')
    plt.plot(validation_steps, validation_losses, label='Validation Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('epoch')
    plt.legend(loc='center left')
    plt.grid(True)
    plt.show()


# Set up lists to store losses for plotting
train_losses = []
train_steps = []
eval_losses = []
eval_steps = []


for step_num, sample in enumerate(train_dataloader):

  model.train()
  input, targets = sample
  logits = model(input)

  logits_view = logits.view(batch_size * config["context_size"], config["vocabulary_size"])
  targets_view = targets.view(batch_size * config["context_size"])
  
  loss = F.cross_entropy(logits_view, targets_view)
  # Backward propagation
  loss.backward()
  # Update model parameters
  optimizer.step()
  # Set to None to reduce memory usage
  optimizer.zero_grad(set_to_none=True)

  # Append training loss
  train_losses.append(loss.item())
  # Append training step
  train_steps.append(step_num)
  
  print(f"Step {step_num}. Loss {loss.item():.3f}")

  if step_num % evaluation_interval == 0:
    print("Demo GPT:\n" + generate_with_prompt(model, tokenizer, "\n"))

    validation_loss = calculate_validation_loss(model, batches_num=10)
    # Append validation loss
    eval_losses.append(validation_loss)
    # Append validation step
    eval_steps.append(step_num)

    print(f"Step {step_num}. Validation loss: {validation_loss:.3f}")

  update_plot(train_losses, train_steps, eval_losses, eval_steps)

Output()

Step 0. Loss 4.301
Demo GPT:

    t a t e  t  t        t t  thi   o  a c i l  a t a     a    r  t I w  t F a   e   g t  h n a  r t
Step 0. Validation loss: 4.706
Step 1. Loss 4.622
Step 2. Loss 4.356
Step 3. Loss 4.387
Step 4. Loss 3.892
Step 5. Loss 3.240
Step 6. Loss 3.043
Step 7. Loss 2.891
Step 8. Loss 2.923
Step 9. Loss 2.873
Step 10. Loss 2.815
Demo GPT:


Thandon wer.
Abe h
d ongl s
Far e,

Sof f RH or:
S
Sd m wh cer mmI meathindoreele yal hr l,d lheI l
Step 10. Validation loss: 2.804
Step 11. Loss 2.798
Step 12. Loss 2.724
Step 13. Loss 2.724
Step 14. Loss 2.688
Step 15. Loss 2.685
Step 16. Loss 2.687
Step 17. Loss 2.672
Step 18. Loss 2.643
Step 19. Loss 2.636
Step 20. Loss 2.620
Demo GPT:

Botise moofotat ds, er intans ks dn wno isthy I d'se he sH'de l hinre n buswe ofornovensale we shadi
Step 20. Validation loss: 2.597
Step 21. Loss 2.600
Step 22. Loss 2.582
Step 23. Loss 2.584
Step 24. Loss 2.576
Step 25. Loss 2.558
Step 26. Loss 2.560
Step 27. Loss 2.553
Step 28. Loss 2.561

Step 231. Loss 2.066
Step 232. Loss 2.094
Step 233. Loss 2.065
Step 234. Loss 2.084
Step 235. Loss 2.065
Step 236. Loss 2.065
Step 237. Loss 2.057
Step 238. Loss 2.058
Step 239. Loss 2.041
Step 240. Loss 2.026
Demo GPT:

Your rius cunariente oncour to l cor,
I'd row nes you, my, my dentur tond entom;
What lay fay and gl
Step 240. Validation loss: 2.131
Step 241. Loss 2.078
Step 242. Loss 2.083
Step 243. Loss 2.054
Step 244. Loss 2.046
Step 245. Loss 2.082
Step 246. Loss 2.051
Step 247. Loss 2.035
Step 248. Loss 2.005
Step 249. Loss 2.016
Step 250. Loss 1.999
Demo GPT:

ICHAR:
I'LINCA:
Sce. ith ame? If leavand, heis geve me?

MOULIUCEST: will pair hord mant,
You te tor
Step 250. Validation loss: 2.088
Step 251. Loss 2.007
Step 252. Loss 2.019
Step 253. Loss 2.019
Step 254. Loss 2.017
Step 255. Loss 2.040
Step 256. Loss 2.013
Step 257. Loss 1.993
Step 258. Loss 2.012
Step 259. Loss 1.979
Step 260. Loss 2.012
Demo GPT:

We morven and me hour hast oreffire:
The reat were if chme'd fuld mas

Step 463. Loss 1.572
Step 464. Loss 1.572
Step 465. Loss 1.592
Step 466. Loss 1.582
Step 467. Loss 1.592
Step 468. Loss 1.612
Step 469. Loss 1.593
Step 470. Loss 1.606
Demo GPT:

That furth the know not your dracklory,
I your strong his perial: you have your word
as in'd a day'd
Step 470. Validation loss: 1.734
Step 471. Loss 1.611
Step 472. Loss 1.605
Step 473. Loss 1.554
Step 474. Loss 1.578
Step 475. Loss 1.612
Step 476. Loss 1.595
Step 477. Loss 1.596
Step 478. Loss 1.569
Step 479. Loss 1.573
Step 480. Loss 1.595
Demo GPT:


VINIUS:
I none.

Let Mursenger:
I poid those, and by for-known, own a give the
again.


ROMEO:

Fay
Step 480. Validation loss: 1.728
Step 481. Loss 1.596
Step 482. Loss 1.602
Step 483. Loss 1.576
Step 484. Loss 1.572
Step 485. Loss 1.607
Step 486. Loss 1.598
Step 487. Loss 1.567
Step 488. Loss 1.573
Step 489. Loss 1.563
Step 490. Loss 1.584
Demo GPT:

He suish?

MARCIUS:
You.

KING EDWARD IV:
Myst?

GLOUCESTER:
AMy soxterlow again who: were no it?

Q
Step 490. 